# 🏠 Regression Model — Prediksi Harga Rumah
**Dataset:** DATA_RUMAH.xlsx | **Target:** HARGA

**Alur Analisis:**
1. Import Library
2. Load & Persiapan Data
3. Exploratory Data Analysis (EDA)
4. Feature Selection & Korelasi
5. Pre-Processing
6. Modelling & Evaluasi

## 1. 📦 Import Library

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.feature_selection import SelectKBest, f_regression

plt.rcParams['figure.figsize'] = (10, 6)
sns.set_style('whitegrid')

## 2. 📂 Load & Persiapan Data

In [4]:
from google.colab import files
uploaded = files.upload()  

df = pd.read_excel('DATA RUMAH.xlsx')
print(f'Shape: {df.shape}')
df.head(10)

ModuleNotFoundError: No module named 'google'

In [ ]:
print('=== INFO DATASET ===')
df.info()
print()
df.describe()

In [ ]:
# Cek missing values & duplikat
print('Missing values:')
print(df.isnull().sum())
print(f'\nDuplikat: {df.duplicated().sum()}')

# Drop kolom tidak relevan
df_clean = df.drop(columns=['NO', 'NAMA RUMAH'])
print(f'\nKolom fitur: {df_clean.columns.tolist()}')

## 3. 📊 Exploratory Data Analysis (EDA)

In [ ]:
# Distribusi HARGA — original vs log transform
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df_clean['HARGA'], bins=40, color='steelblue', edgecolor='white')
axes[0].set_title('Distribusi HARGA (Original)')
axes[0].set_xlabel('Harga (IDR)')

axes[1].hist(np.log1p(df_clean['HARGA']), bins=40, color='coral', edgecolor='white')
axes[1].set_title('Distribusi HARGA (Log Transform)')
axes[1].set_xlabel('Log(Harga)')
plt.tight_layout()
plt.show()

In [ ]:
# Distribusi semua fitur
features = ['LB', 'LT', 'KT', 'KM', 'GRS']
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
for i, col in enumerate(features):
    axes[i].hist(df_clean[col], bins=25, color='mediumpurple', edgecolor='white')
    axes[i].set_title(f'Distribusi {col}')
axes[-1].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# Boxplot untuk deteksi outlier
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
for i, col in enumerate(['HARGA', 'LB', 'LT', 'KT', 'KM', 'GRS']):
    axes[i].boxplot(df_clean[col], patch_artist=True,
                    boxprops=dict(facecolor='lightblue', color='navy'))
    axes[i].set_title(f'Boxplot {col}')
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plot setiap fitur vs HARGA
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()
for i, col in enumerate(features):
    axes[i].scatter(df_clean[col], df_clean['HARGA'], alpha=0.4, color='teal', s=20)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('HARGA')
    axes[i].set_title(f'{col} vs HARGA')
axes[-1].set_visible(False)
plt.tight_layout()
plt.show()

## 4. 🔍 Feature Selection & Analisis Korelasi

In [ ]:
# Heatmap korelasi
corr_matrix = df_clean.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            mask=mask, vmin=-1, vmax=1, linewidths=0.5, square=True)
plt.title('Heatmap Korelasi Antar Fitur', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Korelasi setiap fitur terhadap HARGA
corr_target = df_clean.corr()['HARGA'].drop('HARGA').sort_values(ascending=False)
print('=== KORELASI FITUR vs HARGA ===')
print(corr_target)

colors = ['green' if v > 0 else 'red' for v in corr_target.values]
corr_target.plot(kind='bar', color=colors, edgecolor='black', figsize=(8, 5))
plt.title('Korelasi Fitur vs HARGA')
plt.ylabel('Pearson Correlation')
plt.axhline(0, color='black', linewidth=0.8)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# SelectKBest - F-regression score
selector = SelectKBest(score_func=f_regression, k='all')
selector.fit(df_clean[features], df_clean['HARGA'])

f_scores = pd.DataFrame({
    'Fitur': features,
    'F-Score': selector.scores_,
    'P-Value': selector.pvalues_
}).sort_values('F-Score', ascending=False)

print('=== F-REGRESSION SCORE ===')
print(f_scores.to_string(index=False))

selected_features = f_scores[f_scores['P-Value'] < 0.05]['Fitur'].tolist()
print(f'\nFitur terpilih (p < 0.05): {selected_features}')

## 5. ⚙️ Pre-Processing

In [ ]:
# Hapus outlier ekstrem pada HARGA dengan IQR
Q1 = df_clean['HARGA'].quantile(0.25)
Q3 = df_clean['HARGA'].quantile(0.75)
IQR = Q3 - Q1
df_model = df_clean[
    (df_clean['HARGA'] >= Q1 - 3*IQR) &
    (df_clean['HARGA'] <= Q3 + 3*IQR)
].copy()
print(f'Data sebelum: {len(df_clean)} | Sesudah: {len(df_model)} | Dihapus: {len(df_clean)-len(df_model)}')

In [ ]:
# Log transform HARGA
df_model['LOG_HARGA'] = np.log1p(df_model['HARGA'])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df_model['HARGA'], bins=30, color='steelblue')
axes[0].set_title(f'HARGA Original | Skew: {df_model["HARGA"].skew():.2f}')
axes[1].hist(df_model['LOG_HARGA'], bins=30, color='coral')
axes[1].set_title(f'LOG_HARGA | Skew: {df_model["LOG_HARGA"].skew():.2f}')
plt.tight_layout()
plt.show()

In [ ]:
# Split fitur & target, lalu scaling
X = df_model[selected_features]
y = df_model['LOG_HARGA']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = RobustScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f'Train: {X_train_s.shape} | Test: {X_test_s.shape}')

## 6. 🤖 Modelling & Evaluasi

In [ ]:
def evaluate_model(name, model, X_tr, y_tr, X_te, y_te):
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    y_pred_orig = np.expm1(y_pred)
    y_true_orig = np.expm1(y_te)

    mae  = mean_absolute_error(y_true_orig, y_pred_orig)
    rmse = np.sqrt(mean_squared_error(y_true_orig, y_pred_orig))
    r2   = r2_score(y_te, y_pred)
    cv   = cross_val_score(model, X_tr, y_tr, cv=5, scoring='r2')

    print(f'\n{'='*45}')
    print(f'  MODEL: {name}')
    print(f'{'='*45}')
    print(f'  MAE  : Rp {mae:>15,.0f}')
    print(f'  RMSE : Rp {rmse:>15,.0f}')
    print(f'  R²   : {r2:.4f}')
    print(f'  CV R² (5-fold): {cv.mean():.4f} ± {cv.std():.4f}')
    return {'Model': name, 'MAE': mae, 'RMSE': rmse, 'R2': r2, 'CV_R2': cv.mean()}

results = []

In [ ]:
lr = LinearRegression()
results.append(evaluate_model('Linear Regression', lr, X_train_s, y_train, X_test_s, y_test))

In [ ]:
ridge = Ridge(alpha=10)
results.append(evaluate_model('Ridge Regression', ridge, X_train_s, y_train, X_test_s, y_test))

In [ ]:
lasso = Lasso(alpha=0.001)
results.append(evaluate_model('Lasso Regression', lasso, X_train_s, y_train, X_test_s, y_test))

In [ ]:
rf = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
results.append(evaluate_model('Random Forest', rf, X_train_s, y_train, X_test_s, y_test))

In [ ]:
gb = GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, max_depth=4, random_state=42)
results.append(evaluate_model('Gradient Boosting', gb, X_train_s, y_train, X_test_s, y_test))

In [ ]:
# Perbandingan semua model
df_results = pd.DataFrame(results).sort_values('R2', ascending=False)
print('\n=== PERBANDINGAN SEMUA MODEL ===')
print(df_results[['Model','R2','CV_R2','MAE','RMSE']].to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].barh(df_results['Model'], df_results['R2'], color='steelblue', edgecolor='white')
axes[0].set_title('R² Score per Model')
axes[0].set_xlabel('R²')

axes[1].barh(df_results['Model'], df_results['MAE']/1e9, color='coral', edgecolor='white')
axes[1].set_title('MAE per Model (Miliar IDR)')
axes[1].set_xlabel('MAE (Rp Miliar)')
plt.tight_layout()
plt.show()

In [ ]:
# Actual vs Predicted — Best Model
best_name = df_results.iloc[0]['Model']
model_map = {'Linear Regression': lr, 'Ridge Regression': ridge,
             'Lasso Regression': lasso, 'Random Forest': rf, 'Gradient Boosting': gb}
best_model = model_map[best_name]

y_pred_best = best_model.predict(X_test_s)
y_actual    = np.expm1(y_test)
y_predicted = np.expm1(y_pred_best)

plt.figure(figsize=(8, 8))
plt.scatter(y_actual, y_predicted, alpha=0.4, color='teal', s=25)
lim = [min(y_actual.min(), y_predicted.min()), max(y_actual.max(), y_predicted.max())]
plt.plot(lim, lim, 'r--', linewidth=1.5, label='Perfect Fit')
plt.xlabel('Harga Aktual (IDR)')
plt.ylabel('Harga Prediksi (IDR)')
plt.title(f'Actual vs Predicted — {best_name}')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance / Koefisien
if hasattr(best_model, 'feature_importances_'):
    importances = pd.Series(best_model.feature_importances_, index=selected_features).sort_values()
    importances.plot(kind='barh', color='mediumseagreen', edgecolor='white', figsize=(8, 5))
    plt.title(f'Feature Importance — {best_name}')
    plt.xlabel('Importance Score')
else:
    coef = pd.Series(best_model.coef_, index=selected_features).sort_values()
    coef.plot(kind='barh', color='mediumpurple', edgecolor='white', figsize=(8, 5))
    plt.title(f'Koefisien Fitur — {best_name}')
    plt.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

In [ ]:
# Residual Plot
residuals = y_test.values - y_pred_best

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(y_pred_best, residuals, alpha=0.4, color='orange', s=20)
axes[0].axhline(0, color='red', linestyle='--')
axes[0].set_xlabel('Predicted (log scale)')
axes[0].set_ylabel('Residuals')
axes[0].set_title('Residual Plot')

axes[1].hist(residuals, bins=40, color='orange', edgecolor='white')
axes[1].set_title('Distribusi Residual')
plt.tight_layout()
plt.show()

print(f'\nModel Terbaik: {best_name}')
best_row = df_results.iloc[0]
print(f'R²   : {best_row["R2"]:.4f}')
print(f'MAE  : Rp {best_row["MAE"]:,.0f}')
print(f'RMSE : Rp {best_row["RMSE"]:,.0f}')